# Estimate severity NL
For the new respondents in NL (5th wave), they forgot to collect the perceived damage data. Therefore, we use the data from the first wave to get some equation for the estimation of perceved damage, so that we can estimate the perceived damage for wave 5. 

We base it on the following variables:
- perceived probability
- worry
- flood experience

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [2]:
data_NL_w1 = pd.read_csv('../../../data/raw/SCALAR_Coastal_Longitudinal_Study_Wave_One_NL.csv')

In [3]:
data_NL_w1.head()

,ID,Q0_age,Q0_gender,Q0_postcode,Q0_education_NL,Q0_employment_ID_NL,Q1_home_ID_NL_US,Q2_floors_1,Q2_floors_2,Q2_floors_3,Q3_constr_qual,Q4_home_size_CN_ID_NL,Q5_home_tenure,Q5a_mortgage,Q5b_home_sell,Q6_home_costs,Q7_move_in,Q8_move_out,Q9_easy_leave,Q10_community,Q11_search_improve,Q11_search_social,Q11_search_family,Q11_search_area,Q11_search_job,Q11_search_location,Q11_search_hazard,Q11_search_other,Q11_search_dontknow,Q11a_hazard_type1,Q11a_hazard_type2,Q11a_hazard_type3,Q11a_hazard_type4,Q11a_hazard_type5,Q11a_hazard_type6,Q11a_hazard_type7,Q11a_hazard_type8,Q11a_hazard_type9,Q11a_not_say,Q12_risk_aversion,R01_resilience_1,R01_resilience_2,R01_resilience_3,R01_resilience_4,R01_resilience_5,R01_resilience_6,Q14_early_warn,Q15_responsibility,Q16_compens_gov,Q17_compens_noone,Q17_compens_ins,Q17_compens_owner,Q17_compens_family,Q17_compens_ngo,Q17_compens_other,Q17_compens_dont_know,Q18_flood_exp,Q18a_flood_where,Q18b_flood_year,Q18c_flood_health,Q18ci_flood_work,Q18d_flood_cost,Q20_support_accom,Q21_support_financial,Q22_gov_measures,R02_perc_prob,Q24_perc_prob_change,R03_perc_damage,Q27_perc_prob_30y,R04_perc_health,R05_worry,R06a_media_freq,R06b_social_media_freq,Q31a_media_trust,Q31b_social_media_trust,Q32_climate_belief,Q33_climate_affect,R1a_self_efficacy_SM1,R1a_self_efficacy_SM2,R1a_self_efficacy_SM3,R1a_self_efficacy_SM4,R1a_self_efficacy_SM5,R1a_self_efficacy_SM6,R1a_self_efficacy_SM7,R1b_resp_efficacy_SM1,R1b_resp_efficacy_SM2,R1b_resp_efficacy_SM3,R1b_resp_efficacy_SM4,R1b_resp_efficacy_SM5,R1b_resp_efficacy_SM6,R1b_resp_efficacy_SM7,R1c_perc_cost_SM1,R1c_perc_cost_SM2,R1c_perc_cost_SM3,R1c_perc_cost_SM4,R1c_perc_cost_SM5,R1c_perc_cost_SM6,R1c_perc_cost_SM7,R2_implementation_SM1,R2_implementation_SM2,R2_implementation_SM3,R2_implementation_SM4,R2_implementation_SM5,R2_implementation_SM6,R2_implementation_SM7,Q37_dam_reduction,R1a_self_efficacy_NM1,R1a_self_efficacy_NM2,R1a_self_efficacy_NM3,R1a_self_efficacy_NM4,R1a_self_efficacy_NM5,R1a_self_efficacy_NM6,R1a_self_efficacy_NM7,R1a_self_efficacy_NM8,R1a_self_efficacy_NM9,R1a_self_efficacy_NM10,R1a_self_efficacy_NM11,R1b_resp_efficacy_NM1,R1b_resp_efficacy_NM2,R1b_resp_efficacy_NM3,R1b_resp_efficacy_NM4,R1b_resp_efficacy_NM5,R1b_resp_efficacy_NM6,R1b_resp_efficacy_NM7,R1b_resp_efficacy_NM8,R1b_resp_efficacy_NM9,R1b_resp_efficacy_NM10,R1b_resp_efficacy_NM11,R1c_perc_cost_NM1,R1c_perc_cost_NM2,R1c_perc_cost_NM3,R1c_perc_cost_NM4,R1c_perc_cost_NM5,R1c_perc_cost_NM6,R1c_perc_cost_NM7,R1c_perc_cost_NM8,R1c_perc_cost_NM9,R1c_perc_cost_NM10,R1c_perc_cost_NM11,R2_implementation_NM1,R2_implementation_NM2,R2_implementation_NM3,R2_implementation_NM4,R2_implementation_NM5,R2_implementation_NM6,R2_implementation_NM7,R2_implementation_NM8,R2_implementation_NM9,R2_implementation_NM10,R2_implementation_NM11,Q44_social_expectation,R07_adaptation_others,Q46_dikes_NL,Q47b_industry_type_NL,Q48_business_owner,Q49_self_employed_NL,Q50_employer_size,Q51_unempl_time,Q52_multiple_income,Q52a_income_job,Q53_income_NL,R08_economic_comfort,R09_savings_change,R09a_savings_change_frac,Q56_savings_change_future,Q56a_savings_change_future_frac,Q57_hh_size_NL,Q58_savings,Q59_disability,Q60_child,Q60_elder,Q60_no,Q60_not_say,Q61_single_parent
0,0,2,2,3193.0,3,2,1,1.0,0.0,1.0,2,5,1,NaN,NaN,1300,2014,3,2,0,1,0,0,0,0,0,0,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,2,2,2,3,3,2,0,3,81.0,0,1,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,1,1,2,5,1,4,40.0,2,2,4,3,4,1,2,3,1,1,1,1,1,1,1,4,4,4,4,4,4,4,4,4,3,4,4,4,4,6,6,6,6,6,6,6,NaN,5,5,1,5,5,5,5,3,4,4,4,4,3,4,3,3,3,4,3,3,3,3,1,3,3,3,3,3,1,3,2,3,3,1,6,6,6,6,6,1,6,6,6,6,1,1,0,97.0,1,1,6.0,2.0,1,4.0,3,4,1,40.0,3,25.0,2,7,0,0,0,1,0,0
1,1,4,1,3055.0,3,1,2,1.0,1.0,1.0,3,5,2,2.0,400000.0,900,2008,3,3,0,0,1,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,3,3,2,2,2,4,0,1,50.0,0,1,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,0,1,3,6,2,4,1.0,3,2,3,3,3,3,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,5,5,5,5,5,5,5,6,6,6,6,6,6,6,NaN,5,5,5,5,5,5,1,1,1,1,1,1,1,2,2,1,1,2,2,2,4,3,4,3,3,3,4,4,5,5,5,5,5,2,6

In [4]:
data_for_regression = data_NL_w1[['Q18_flood_exp', 'R03_perc_damage', 'R02_perc_prob', 'R05_worry']]
data_for_regression.describe()

,Q18_flood_exp,R03_perc_damage,R02_perc_prob,R05_worry
count,1251.000000,1251.000000,1251.000000,1251.000000
mean,0.119904,11.986411,20.056755,2.054357
std,0.324979,27.530155,36.162577,1.130771
min,0.000000,1.000000,1.000000,1.000000
25%,0.000000,3.000000,2.000000,1.000000
50%,0.000000,3.000000,4.000000,2.000000
75%,0.000000,4.000000,6.000000,3.000000
max,1.000000,98.000000,98.000000,5.000000


In [5]:
data_for_regression = data_for_regression[(data_for_regression.R02_perc_prob != 95) &
                                           (data_for_regression.R02_perc_prob != 97) &
                                            (data_for_regression.R02_perc_prob != 98) &
                                            (data_for_regression.R03_perc_damage != 95) &
                                           (data_for_regression.R03_perc_damage != 97) &
                                            (data_for_regression.R03_perc_damage != 98)]
data_for_regression.describe()

,Q18_flood_exp,R03_perc_damage,R02_perc_prob,R05_worry
count,968.000000,968.000000,968.000000,968.000000
mean,0.136364,3.150826,3.393595,2.028926
std,0.343352,1.148705,1.974774,1.120086
min,0.000000,1.000000,1.000000,1.000000
25%,0.000000,2.000000,2.000000,1.000000
50%,0.000000,3.000000,3.000000,2.000000
75%,0.000000,4.000000,5.000000,3.000000
max,1.000000,5.000000,9.000000,5.000000


In [6]:
data_for_regression.dtypes

Q18_flood_exp      int64
R03_perc_damage    int64
R02_perc_prob      int64
R05_worry          int64
dtype: object

In [7]:
scaler = MinMaxScaler()
data_for_regression[['Q18_flood_exp', 'R03_perc_damage', 'R02_perc_prob', 'R05_worry']] = scaler.fit_transform(data_for_regression[['Q18_flood_exp', 'R03_perc_damage', 'R02_perc_prob', 'R05_worry']])
data_for_regression.head()

,Q18_flood_exp,R03_perc_damage,R02_perc_prob,R05_worry
0,0.0,0.75,0.500,0.25
1,0.0,0.75,0.625,0.25
2,0.0,1.00,0.250,0.50
3,0.0,0.00,0.000,0.25
4,0.0,0.50,0.500,0.25


In [8]:
data_for_regression.describe()

,Q18_flood_exp,R03_perc_damage,R02_perc_prob,R05_worry
count,968.000000,968.000000,968.000000,968.000000
mean,0.136364,0.537707,0.299199,0.257231
std,0.343352,0.287176,0.246847,0.280021
min,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.250000,0.125000,0.000000
50%,0.000000,0.500000,0.250000,0.250000
75%,0.000000,0.750000,0.500000,0.500000
max,1.000000,1.000000,1.000000,1.000000


In [9]:
X = data_for_regression[['R02_perc_prob', 'R05_worry', 'Q18_flood_exp']]
y = data_for_regression[['R03_perc_damage']]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = LinearRegression()
model.fit(X_train, y_train)

# Evaluate
predictions = model.predict(X_test)
print("RMSE:", np.sqrt(mean_squared_error(y_test, predictions)))

# Extract weights (coefficients) and intercept
weights = model.coef_[0]  # Coefficients for each feature
intercept = model.intercept_[0]  # Intercept

print(weights, intercept)
# Print the logistic regression equation
feature_names = X_train.columns
equation = f"per_damage = {intercept:.4f} " + " + ".join([f"({coef:.4f} * {name})" for coef, name in zip(weights, feature_names)])

print("\nLinear Regression Equation:")
print(equation)
print("\nFeature Weights:", dict(zip(feature_names, weights)))
print("Intercept:", intercept)

RMSE: 0.27391822086525935
[ 0.22745863  0.17538449 -0.10958098] 0.4360333859563724

Linear Regression Equation:
per_damage = 0.4360 (0.2275 * R02_perc_prob) + (0.1754 * R05_worry) + (-0.1096 * Q18_flood_exp)

Feature Weights: {'R02_perc_prob': 0.22745862825043375, 'R05_worry': 0.17538449419988458, 'Q18_flood_exp': -0.10958098206299823}
Intercept: 0.4360333859563724


Rule of thumb: (Deepseek)
- Excellent: RMSE < 0.5 × standard deviation
- Good: RMSE < 0.7 × standard deviation
- Fair: RMSE < 1 × standard deviation
- Poor: RMSE > 1 × standard deviation

std = 0.28 so not great regression

just gonna go with this for now. Not sure how to make it better...

In [10]:
# Create polynomial regression pipeline
degree = 2 # degrees do pretty much nothing for model fit
poly_model = make_pipeline(
    PolynomialFeatures(degree=degree, include_bias=False),
    LinearRegression()
)

# Fit the model
poly_model.fit(X_train, y_train)

# Evaluate
predictions = poly_model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
print(f"RMSE: {rmse:.4f}")

# Extract coefficients and feature names
linear_reg = poly_model.named_steps['linearregression']
poly_features = poly_model.named_steps['polynomialfeatures']

# Get feature names for polynomial terms
feature_names = poly_features.get_feature_names_out(input_features=X_train.columns)

# Get coefficients
intercept = linear_reg.intercept_
weights = linear_reg.coef_[0]
print('Intercept:', intercept)
print('Coefficients:', weights)
print('Variables:', feature_names)

RMSE: 0.2698
Intercept: [0.39394693]
Coefficients: [ 0.50937906  0.44099468 -0.03715383 -0.39922436 -0.04400773 -0.05989569
 -0.30487762 -0.07125828 -0.03715383]
Variables: ['R02_perc_prob' 'R05_worry' 'Q18_flood_exp' 'R02_perc_prob^2'
 'R02_perc_prob R05_worry' 'R02_perc_prob Q18_flood_exp' 'R05_worry^2'
 'R05_worry Q18_flood_exp' 'Q18_flood_exp^2']
